# Whole State Space vs Additive Periodic+Whole Time

This notebook combines two result tables:
- full state-space search results
- periodic-only search results

Additive-time rule:
1. start from periodic search time
2. if periodic says unsafe, stop there
3. if periodic says safe, add full-state-space search time

In [1]:
import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["text.usetex"] = True
plt.rcParams.update({"figure.titlesize": "small"})
plt.rcParams.update({"axes.titlesize": "small"})
plt.rcParams.update({"axes.labelsize": "small"})
plt.rcParams.update({"ytick.labelsize": "small"})
plt.rcParams.update({"xtick.labelsize": "small"})
plt.rcParams.update({"legend.fontsize": "small"})

In [5]:
FULL_PATH = "./results/results-mcs_schedulability-2026-03-28_20-57-48.csv"
PERIODIC_PATH = "./results/results-mcs_schedulability-2026-04-15_20-29-49.csv"

df_full = pd.read_csv(FULL_PATH, sep=";")
df_periodic = pd.read_csv(PERIODIC_PATH, sep=";")

print(f"full rows: {len(df_full)}")
print(f"periodic rows: {len(df_periodic)}")

full rows: 22000
periodic rows: 22000


In [6]:
# Build merge keys from common identifier columns.
candidate_keys = [
    "taskset_position",
    "U",
    "Uv",
    "nbt",
    "EDFVD_test",
    "probability_of_HI",
    "min_period",
    "max_period",
]

merge_keys = [
    c for c in candidate_keys if c in df_full.columns and c in df_periodic.columns
]

if not merge_keys:
    raise ValueError("No common merge keys found between the two tables.")

df_full_use = df_full[merge_keys + ["duration_ns"]].rename(
    columns={"duration_ns": "duration_ns_full"}
)

df_periodic_use = df_periodic[merge_keys + ["duration_ns", "is_safe"]].rename(
    columns={"duration_ns": "duration_ns_periodic", "is_safe": "is_safe_periodic"}
)

df_merged = df_periodic_use.merge(df_full_use, on=merge_keys, how="inner")

if df_merged.empty:
    raise ValueError(
        "Merge produced zero rows. Adjust merge_keys or input files to matching experiments."
    )

df_merged["is_safe_periodic"] = df_merged["is_safe_periodic"].astype(str).str.lower() == "true"
df_merged["duration_s_periodic"] = df_merged["duration_ns_periodic"] / 1e9
df_merged["duration_s_full"] = df_merged["duration_ns_full"] / 1e9

df_merged["duration_s_additive"] = np.where(
    df_merged["is_safe_periodic"],
    df_merged["duration_s_periodic"] + df_merged["duration_s_full"],
    df_merged["duration_s_periodic"],
)

df_merged["additive_case"] = np.where(
    df_merged["is_safe_periodic"],
    "periodic-safe-then-full",
    "periodic-unsafe-stop",
)

print(f"merge keys: {merge_keys}")
print(f"matched rows: {len(df_merged)}")
df_merged.head()

merge keys: ['taskset_position', 'U', 'Uv', 'nbt', 'EDFVD_test', 'probability_of_HI', 'min_period', 'max_period']
matched rows: 44000


,taskset_position,U,Uv,nbt,EDFVD_test,probability_of_HI,min_period,max_period,duration_ns_periodic,is_safe_periodic,duration_ns_full,duration_s_periodic,duration_s_full,duration_s_additive,additive_case
0,2,0.5,0.497348,5.0,1.0,0.5,5.0,30.0,73969,True,27461480,0.000074,0.027461,0.027535,periodic-safe-then-full
1,2,0.5,0.497348,5.0,1.0,0.5,5.0,30.0,73969,True,70838437,0.000074,0.070838,0.070912,periodic-safe-then-full
2,4,0.5,0.497891,5.0,1.0,0.5,5.0,30.0,164438,True,27549928,0.000164,0.027550,0.027714,periodic-safe-then-full
3,4,0.5,0.497891,5.0,1.0,0.5,5.0,30.0,164438,True,131990745,0.000164,0.131991,0.132155,periodic-safe-then-full
4,10,0.5,0.500592,5.0,1.0,0.5,5.0,30.0,210997,True,1653223224,0.000211,1.653223,1.653434,periodic-safe-then-full


In [7]:
comparison_cols = merge_keys + [
    "is_safe_periodic",
    "duration_s_periodic",
    "duration_s_full",
    "duration_s_additive",
    "additive_case",
]

df_comparison = df_merged[comparison_cols].sort_values(
    [c for c in ["U", "taskset_position"] if c in comparison_cols]
)

df_comparison

,taskset_position,U,Uv,nbt,EDFVD_test,probability_of_HI,min_period,max_period,is_safe_periodic,duration_s_periodic,duration_s_full,duration_s_additive,additive_case
8,0,0.5,0.502598,5.0,1.0,0.5,5.0,30.0,True,0.000185,0.065697,0.065882,periodic-safe-then-full
9,0,0.5,0.502598,5.0,1.0,0.5,5.0,30.0,True,0.000185,0.288559,0.288744,periodic-safe-then-full
21990,0,0.5,0.502598,5.0,1.0,0.5,5.0,30.0,True,0.000143,0.065697,0.065840,periodic-safe-then-full
21991,0,0.5,0.502598,5.0,1.0,0.5,5.0,30.0,True,0.000143,0.288559,0.288702,periodic-safe-then-full
6,1,0.5,0.498661,5.0,1.0,0.5,5.0,30.0,True,0.000121,0.225581,0.225702,periodic-safe-then-full
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43985,10998,1.0,0.998864,5.0,0.0,0.5,5.0,30.0,False,0.000052,0.127140,0.000052,periodic-unsafe-stop
22002,10999,1.0,0.995629,5.0,0.0,0.5,5.0,30.0,False,0.000006,0.000102,0.000006,periodic-unsafe-stop
22003,10999,1.0,0.995629,5.0,0.0,0.5,5.0,30.0,False,0.000006,1.031503,0.000006,periodic-unsafe-stop
43974,10999,1.0,0.995629,5.0,0.0,0.5,5.0,30.0,True,0.000633,0.000102,0.000735,periodic-safe-then-full


In [8]:
out_path = "./results/whole_vs_additive_comparison.csv"
df_comparison.to_csv(out_path, sep=";", index=False)
print(f"saved: {out_path}")

saved: ./results/whole_vs_additive_comparison.csv


In [9]:
df_plot = pd.concat(
    [
        df_merged.loc[:, ["U", "duration_s_full"]].rename(
            columns={"duration_s_full": "duration_s"}
        ).assign(approach="Whole state-space"),
        df_merged.loc[:, ["U", "duration_s_additive"]].rename(
            columns={"duration_s_additive": "duration_s"}
        ).assign(approach="Additive periodic+whole"),
    ],
    ignore_index=True,
)

f, ax = plt.subplots(figsize=(3.5, 3.5))
sns.lineplot(
    data=df_plot.sort_values("U"),
    x="U",
    y="duration_s",
    hue="approach",
    marker="o",
    ax=ax,
)

ax.set_ylabel("Time taken (s)")
ax.set_xlabel("Average utilisation ($U^*$)")
ax.set_title(r"Whole vs additive execution time")
sns.move_legend(ax, "upper left")
f.tight_layout()

/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/_base.py:949: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.1

In [10]:
f2, ax2 = plt.subplots(figsize=(3.5, 3.5))

sns.scatterplot(
    data=df_merged,
    x="duration_s_full",
    y="duration_s_additive",
    hue="additive_case",
    ax=ax2,
)

ax2.set(xscale="log", yscale="log")
lims = [
    min(df_merged["duration_s_full"].min(), df_merged["duration_s_additive"].min()),
    max(df_merged["duration_s_full"].max(), df_merged["duration_s_additive"].max()),
]
ax2.plot(lims, lims, "k--", alpha=0.75, zorder=1)
ax2.set(xlim=lims, ylim=lims)
ax2.set_xlabel("Whole state-space time (s)")
ax2.set_ylabel("Additive time (s)")
ax2.set_title("Per-taskset time comparison")
f2.tight_layout()

In [11]:
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
f.savefig(f"./figs/{ts}_whole_vs_additive_u.pdf", bbox_inches="tight")
f.savefig("./figs/whole_vs_additive_u.pdf", bbox_inches="tight")
f2.savefig(f"./figs/{ts}_whole_vs_additive_scatter.pdf", bbox_inches="tight")
f2.savefig("./figs/whole_vs_additive_scatter.pdf", bbox_inches="tight")